In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from py_clob_client_v2 import ApiCreds, ClobClient, SignatureTypeV2

secrets_path = Path("execution_engine/secrets.env")
if secrets_path.exists():
    load_dotenv(secrets_path)
else:
    load_dotenv()

host = os.getenv("CLOB_API_URL", "https://clob.polymarket.com")
chain_id = int(os.getenv("CHAIN_ID", "137"))
private_key = os.environ["POLYMARKET_PRIVATE_KEY"]
deposit_wallet_address = os.getenv("DEPOSIT_WALLET_ADDRESS")

# Derive credentials from the EOA/session signer that owns the deposit wallet.
temp_client = ClobClient(
    host=host,
    chain_id=chain_id,
    key=private_key,
)
creds = temp_client.create_or_derive_api_key()

# Validate the credentials in the same deposit-wallet trading context used live.
if deposit_wallet_address:
    trading_client = ClobClient(
        host=host,
        chain_id=chain_id,
        key=private_key,
        creds=ApiCreds(
            api_key=creds.api_key,
            api_secret=creds.api_secret,
            api_passphrase=creds.api_passphrase,
        ),
        signature_type=SignatureTypeV2.POLY_1271,
        funder=deposit_wallet_address,
    )
    builder = trading_client.builder
    print("signature_type=" + str(int(builder.signature_type)))
    print("funder_present=" + str(bool(builder.funder)))
else:
    print("DEPOSIT_WALLET_ADDRESS is not set; generated credentials only.")

print("CLOB_API_KEY=" + creds.api_key)
print("CLOB_SECRET=" + creds.api_secret)
print("CLOB_PASS_PHRASE=" + creds.api_passphrase)
